# Chapter 3: Activation Functions, Architecture, and Optimization

> Source PDF: `03_Activation_Functions.pdf`  
> Instructors: Maham Faisal Khan and Thomas Hossler

## Learning Objectives
By the end of this notebook, you will be able to:
- Explain limitations of sigmoid and softmax in hidden layers.
- Use ReLU and Leaky ReLU activations.
- Reason about layers, neurons, and model capacity.
- Count trainable parameters in a neural network.
- Explain the effects of learning rate and momentum.
- Initialize layer weights.
- Describe transfer learning and fine-tuning.


In [ ]:
# Core imports used throughout this notebook
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim


## 3.1 Limitations of Sigmoid and Softmax

Sigmoid and softmax are useful at output layers, but they can cause trouble inside deep networks.

| Property | Effect |
|---|---|
| Outputs are bounded | Useful for probabilities. |
| Gradients approach zero for very low/high inputs | Can slow or stop learning. |
| Saturation | Leads to vanishing gradients during backpropagation. |


## 3.2 Introducing ReLU

The Rectified Linear Unit is defined as:

$$f(x) = \max(x, 0)$$

| Input | ReLU Output |
|---|---|
| Positive | Same as input |
| Zero | Zero |
| Negative | Zero |


In [ ]:
# ReLU in PyTorch
relu = nn.ReLU()

x = torch.tensor([-3.0, -1.0, 0.0, 1.0, 3.0])
print("Input:", x)
print("ReLU output:", relu(x))


## 3.3 Introducing Leaky ReLU

Leaky ReLU behaves like ReLU for positive inputs, but for negative inputs it multiplies by a small coefficient.

| Activation | Negative Inputs | Positive Inputs |
|---|---|---|
| ReLU | `0` | `x` |
| Leaky ReLU | `negative_slope * x` | `x` |


In [ ]:
# Leaky ReLU in PyTorch
leaky_relu = nn.LeakyReLU(negative_slope=0.05)

x = torch.tensor([-3.0, -1.0, 0.0, 1.0, 3.0])
print("Input:", x)
print("Leaky ReLU output:", leaky_relu(x))


## 3.4 A Deeper Dive into Neural Network Architecture

Linear layers are fully connected: every neuron in a layer is connected to every neuron in the previous layer.

A neuron in a linear layer:
- computes a linear operation using all outputs from the previous layer;
- contains `N + 1` learnable parameters, where `N` is the previous layer size;
- uses the extra `1` as the bias term.

| Layer Type | Role |
|---|---|
| Input layer | Receives features. |
| Hidden layer | Learns intermediate representations. |
| Output layer | Produces final scores or predictions. |


## 3.5 Tweaking the Number of Hidden Layers

Input and output dimensions are fixed by the problem. Hidden layers can be adjusted to change model capacity.

> 💡 More hidden layers usually mean more parameters and more capacity, but also a higher risk of overfitting.


In [ ]:
n_features = 8
n_classes = 3

model = nn.Sequential(
    nn.Linear(n_features, 8),
    nn.Linear(8, 4),
    nn.Linear(4, n_classes)
)

print(model)


## 3.6 Counting the Number of Parameters

For this model:

```python
model = nn.Sequential(
    nn.Linear(8, 4),
    nn.Linear(4, 2)
)
```

| Layer | Calculation | Parameters |
|---|---:|---:|
| `Linear(8, 4)` | `4 * (8 + 1)` | `36` |
| `Linear(4, 2)` | `2 * (4 + 1)` | `10` |
| Total |  | `46` |


In [ ]:
model = nn.Sequential(
    nn.Linear(8, 4),
    nn.Linear(4, 2)
)

total = 0
for parameter in model.parameters():
    total += parameter.numel()

print(total)


## 3.7 Learning Rate and Momentum

Training a neural network is an optimization problem. SGD updates parameters using gradients.

| Hyperparameter | Controls | Poor Values Can Cause |
|---|---|---|
| Learning rate | Step size | Slow training or unstable performance |
| Momentum | Optimizer inertia | Getting stuck or overshooting |


In [ ]:
sgd = optim.SGD(model.parameters(), lr=0.01, momentum=0.95)
print(sgd)


### Impact of Learning Rate and Momentum

| Setting | Behavior |
|---|---|
| Optimal learning rate | Learns efficiently and converges well. |
| Too small learning rate | Training takes too long. |
| Too high learning rate | Loss may bounce around or diverge. |
| `momentum=0` | Pure SGD; may move slowly through difficult surfaces. |
| `momentum=0.85` to `0.99` | Common useful range. |

The PDF examples compare optimization after 100 steps:

| Setting | Reported Result |
|---|---|
| `lr=0.01`, `momentum=0` | minimum around `x=-1.23`, `y=-0.14` |
| `lr=0.01`, `momentum=0.9` | minimum around `x=0.92`, `y=-2.04` |


## 3.8 Layer Initialization

Layer weights are initialized to small values. Good initialization helps prevent activations from exploding or vanishing.


In [ ]:
import torch.nn as nn

layer = nn.Linear(64, 128)
print(layer.weight.min(), layer.weight.max())


In [ ]:
import torch.nn as nn

layer = nn.Linear(64, 128)
nn.init.uniform_(layer.weight)
print(layer.weight.min(), layer.weight.max())


> 📝 PyTorch initialization functions usually end with an underscore, such as `uniform_`, because they modify tensors in place.


## 3.9 Transfer Learning and Fine-Tuning

**Transfer learning** means reusing a model trained on one task for a second similar task to accelerate training.

Example from the PDF: reuse a model trained on US data scientist salaries for a smaller European salary dataset.


In [ ]:
import torch

layer = nn.Linear(64, 128)
torch.save(layer, "layer.pth")
new_layer = torch.load("layer.pth", weights_only=False)

print(type(new_layer))


### Fine-Tuning

Fine-tuning is a type of transfer learning. Common choices include using a smaller learning rate, freezing some layers, and training layers closer to the output.


In [ ]:
import torch.nn as nn

model = nn.Sequential(
    nn.Linear(64, 128),
    nn.Linear(128, 256)
)

for name, param in model.named_parameters():
    if name == "0.weight":
        param.requires_grad = False

for name, param in model.named_parameters():
    print(name, "requires_grad=", param.requires_grad)


## Chapter Summary

| Topic | Key Point |
|---|---|
| Sigmoid/softmax limitations | Can saturate and cause vanishing gradients. |
| ReLU | Strong default activation for hidden layers. |
| Leaky ReLU | Keeps a small gradient for negative inputs. |
| Hidden layers | Increase capacity but also overfitting risk. |
| Parameter count | Each neuron has weights plus a bias. |
| Learning rate | Controls step size. |
| Momentum | Adds inertia to optimization. |
| Transfer learning | Reuses learned representations. |

✅ **PDF coverage:** sigmoid/softmax limitations, ReLU, Leaky ReLU, neurons and layer naming, hidden layer tuning, parameter counting, SGD learning rate and momentum, layer initialization, model saving/loading, transfer learning, and freezing parameters for fine-tuning.
